# 14 — Gold DLT: Aggregates

## Configuration

In [0]:
# PDF's retail examples ("sales by customer/category/region", "top 10
# customers by spend", "monthly sales trend") mapped to VStone's domain:
#   agg_monthly_street_trend      -> monthly sales trend
#   agg_top10_streets_by_pollution -> top 10 customers by spend
#   agg_traffic_by_location_daily -> sales by region
#   agg_citizen_reports_by_street  -> extra: ties fact_citizen_reports into
#                                     the aggregate layer too
# All join back to dims via the INT natural/surrogate keys (street_id,
# location) — no string columns used for grouping without first resolving
# through a dim join, matching the reference project's convention.

import dlt
from pyspark.sql import functions as F

GOLD_PROPS = {
    "quality": "gold",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}


def _active_dim(name, cols):
    return dlt.read(name).filter(F.col("__END_AT").isNull()).select(*cols).distinct()

## AGG 1: Monthly Street Trend

In [0]:
@dlt.table(
    name="agg_monthly_street_trend",
    comment="Gold Agg: monthly avg noise/pollution/raining by street. "
            "street_name resolved via dim_street join on street_id (INT). "
            "This is the table the Genie space is built on. Audit: gold_load_dt.",
    table_properties={**GOLD_PROPS, "type": "aggregate"},
)
def agg_monthly_street_trend():
    fact = dlt.read("fact_street_readings")
    dim_street = _active_dim("dim_street", ["street_id", "street_name", "danger_score"])
    return (
        fact.join(dim_street, on="street_id", how="left")
        .withColumn("month_year", F.date_format("reading_date", "yyyy-MM"))
        .groupBy("month_year", "street_id", "street_name", "danger_score")
        .agg(
            F.count("*").alias("reading_count"),
            F.round(F.avg("noise"), 4).alias("avg_noise"),
            F.round(F.avg("pollution"), 4).alias("avg_pollution"),
            F.round(F.avg("raining_clipped"), 2).alias("avg_raining_pct"),
        )
        .orderBy("month_year", "street_id")
        .withColumn("gold_load_dt", F.current_timestamp())
    )

## AGG 2: Top 10 Streets by Pollution

In [0]:
@dlt.table(
    name="agg_top10_streets_by_pollution",
    comment="Gold Agg: top 10 streets by average pollution (the VStone equivalent of "
            "'top 10 customers by spend'). street_name via dim_street join. Audit: gold_load_dt.",
    table_properties={**GOLD_PROPS, "type": "aggregate"},
)
def agg_top10_streets_by_pollution():
    fact = dlt.read("fact_street_readings")
    dim_street = _active_dim("dim_street", ["street_id", "street_name"])
    return (
        fact.join(dim_street, on="street_id", how="left")
        .groupBy("street_id", "street_name")
        .agg(
            F.round(F.avg("pollution"), 4).alias("avg_pollution"),
            F.round(F.avg("noise"), 4).alias("avg_noise"),
            F.count("*").alias("reading_count"),
        )
        .orderBy(F.desc("avg_pollution"))
        .limit(10)
        .withColumn("gold_load_dt", F.current_timestamp())
    )


## AGG 3: Daily Traffic by Location

In [0]:
@dlt.table(
    name="agg_traffic_by_location_daily",
    comment="Gold Agg: daily vehicle enter/exit totals by sensor node. "
            "location resolved via dim_node_location join. Audit: gold_load_dt.",
    table_properties={**GOLD_PROPS, "type": "aggregate"},
)
def agg_traffic_by_location_daily():
    fact = dlt.read("fact_traffic_counts")
    dim_loc = _active_dim("dim_node_location", ["location", "latitude", "longitude"])
    return (
        fact.join(dim_loc, on="location", how="left")
        .groupBy("reading_date", "location", "latitude", "longitude")
        .agg(
            F.sum("enter_count").alias("total_enter"),
            F.sum("exit_count").alias("total_exit"),
            F.count("*").alias("reading_count"),
        )
        .orderBy("reading_date", "location")
        .withColumn("gold_load_dt", F.current_timestamp())
    )


## AGG 4: Citizen Reports by Street

In [0]:

@dlt.table(
    name="agg_citizen_reports_by_street",
    comment="Gold Agg: citizen report volume by street and month. Only counts reports where "
            "street_id resolved (fact_citizen_reports.street_id IS NOT NULL — see that table's "
            "best-effort text-match caveat). Audit: gold_load_dt.",
    table_properties={**GOLD_PROPS, "type": "aggregate"},
)
def agg_citizen_reports_by_street():
    fact = dlt.read("fact_citizen_reports").filter(F.col("street_id").isNotNull())
    dim_street = _active_dim("dim_street", ["street_id", "street_name"])
    return (
        fact.join(dim_street, on="street_id", how="left")
        .withColumn("month_year", F.date_format("message_date", "yyyy-MM"))
        .groupBy("month_year", "street_id", "street_name")
        .agg(F.count("*").alias("report_count"))
        .orderBy("month_year", F.desc("report_count"))
        .withColumn("gold_load_dt", F.current_timestamp())
    )
